# Qwen3-TTS Voice Cloner & Dataset Generator (Full Drive Integration)
Use this notebook to clone a voice and generate a dataset for training models like **Piper**.

### 🚀 Setup and Requirements
1. Ensure you are using a **GPU** (Edit -> Notebook settings -> T4 GPU).
2. **Upload** your `reference.wav`, `transcript.txt`, and `sentences.txt` to your **Google Drive** first (e.g., in a folder named `tts_input`).

In [ ]:
# @title 0. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1. Install Dependencies
!apt-get install -y sox libsox-fmt-all
!pip install -U qwen-tts torch soundfile librosa huggingface_hub tqdm

try:
    print("Installing build tools...")
    !pip install ninja packaging
    print("Attempting to install flash-attn...")
    !pip install flash-attn --no-build-isolation
except:
    print("⚠️ FlashAttention installation failed. Proceeding with standard mode.")

In [ ]:
# @title 2. Load Model
import torch
import numpy as np
from qwen_tts import Qwen3TTSModel
import soundfile as sf
import os
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map=device, 
    dtype=torch.bfloat16 if device == "cuda" else torch.float32
)

### 🎙️ Configuration
Point to your files in Google Drive. Typical path: `/content/drive/MyDrive/YOUR_FOLDER/...`

In [ ]:
# @title 3. Setup Voice Clone (From Drive)
ref_audio = "/content/drive/MyDrive/tts_input/reference.wav" # @param {type:"string"}
ref_text_file = "/content/drive/MyDrive/tts_input/transcript.txt" # @param {type:"string"}

if not os.path.exists(ref_audio):
    print(f"❌ Reference audio not found. Please check the path.")
elif not os.path.exists(ref_text_file):
    print(f"❌ Reference transcript file not found.")
else:
    with open(ref_text_file, "r") as f:
        ref_text = f.read().strip()
    
    clone_prompt = model.create_voice_clone_prompt(
        ref_audio=ref_audio,
        ref_text=ref_text,
        x_vector_only_mode=False
    )
    print("✅ Voice clone ready!")

In [ ]:
# @title 4. Batch Generate Samples (Drive Sync)
import numpy as np
import librosa
import soundfile as sf

sentences_file = "/content/drive/MyDrive/tts_input/sentences.txt" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/piper_dataset" # @param {type:"string"}
language = "English" # @param ["English", "Chinese", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"]

os.makedirs(f"{output_dir}/wavs", exist_ok=True)
metadata_path = os.path.join(output_dir, "metadata.csv")

existing_indices = set()
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        for line in f:
            if "|" in line:
                filename = line.split("|")[0]
                try: existing_indices.add(int(filename.split(".")[0]))
                except: pass
    print(f"📌 Found {len(existing_indices)} existing samples. Skipping...")

if not os.path.exists(sentences_file):
    print(f"❌ Sentences file not found at {sentences_file}.")
else:
    with open(sentences_file, "r") as f:
        sentences = [line.strip() for line in f if line.strip()]

    print(f"🚀 Generating missing samples (Total: {len(sentences)})...")
    for i, text in enumerate(tqdm(sentences)):
        if i in existing_indices: continue
        
        filename = f"{i:04d}.wav"
        save_path = os.path.join(output_dir, "wavs", filename)
        
        if os.path.exists(save_path):
            with open(metadata_path, "a") as f:
                f.write(f"{filename}|{text}\n")
            continue

        wavs, sr = model.generate_voice_clone(text=text, language=language, voice_clone_prompt=clone_prompt)
        audio_data = wavs[0]
        if hasattr(audio_data, "cpu"): audio_data = audio_data.cpu().numpy()
        if np.abs(audio_data).max() > 0: audio_data = audio_data / np.abs(audio_data).max() * 0.9
        
        audio_resampled = librosa.resample(audio_data, orig_sr=sr, target_sr=22050)
        sf.write(save_path, audio_resampled, 22050)
        
        with open(metadata_path, "a") as f:
            f.write(f"{filename}|{text}\n")

    print(f"\n✅ Done! Files are at: {output_dir}")